In [ ]:
!pip install pyhealth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 808.3 kB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 622.3/622.3 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 205.4/205.4 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 426.4/426.4 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB

In [ ]:
import tempfile

from pyhealth.datasets import MIMIC3Dataset
from pyhealth.datasets import split_by_patient, get_dataloader
from pyhealth.models import Transformer
from pyhealth.tasks import DrugRecommendationMIMIC3
from pyhealth.trainer import Trainer
import torch

from google.colab import drive
drive.mount('/content/drive')

if __name__ == "__main__":
    # STEP 1: load data
    base_dataset = MIMIC3Dataset(
        root="/content/drive/MyDrive/mimic-iii-clinical-database-1.4",
        tables=["DIAGNOSES_ICD", "PROCEDURES_ICD", "PRESCRIPTIONS"],
        cache_dir=tempfile.TemporaryDirectory().name,
        dev=False,
    )
    base_dataset.stats()

    # STEP 2: set task
    task = DrugRecommendationMIMIC3()
    print(task.input_schema)
    print(task.output_schema)
    sample_dataset = base_dataset.set_task(task)

    train_dataset, val_dataset, test_dataset = split_by_patient(
        sample_dataset, [0.8, 0.1, 0.1]
    )
    train_dataloader = get_dataloader(train_dataset, batch_size=32, shuffle=True)
    val_dataloader = get_dataloader(val_dataset, batch_size=32, shuffle=False)
    test_dataloader = get_dataloader(test_dataset, batch_size=32, shuffle=False)

    # STEP 3: define model
    model = Transformer(
        dataset=sample_dataset,
    )

    # STEP 4: define trainer
    trainer = Trainer(
        model=model,
        metrics=["jaccard_samples", "f1_samples", "pr_auc_samples"],
    )

    trainer.train(
        train_dataloader=train_dataloader,
        val_dataloader=val_dataloader,
        epochs=1,
        monitor="pr_auc_samples",
    )

    # STEP 5: evaluate
    print(trainer.evaluate(test_dataloader))

    #precision@k and recall@k metrics
    model.eval()

    y_true = []
    y_pred = []

    with torch.no_grad():
        for batch in test_dataloader:
            output = model(**batch)

            y_true.append(output["y_true"].cpu())
            y_pred.append(output["y_prob"].cpu())

    y_true = torch.cat(y_true)
    y_pred = torch.cat(y_pred)

    def precision_at_k(y_true, y_pred, k):
      topk = torch.topk(y_pred, k, dim=1).indices

      precisions = []

      for i in range(y_true.shape[0]):
          true_set = set(torch.where(y_true[i] == 1)[0].tolist())
          pred_set = set(topk[i].tolist())

          precisions.append(len(true_set & pred_set) / k)

      return sum(precisions) / len(precisions)

    def recall_at_k(y_true, y_pred, k):
      topk = torch.topk(y_pred, k, dim=1).indices

      recalls = []

      for i in range(y_true.shape[0]):
          true_set = set(torch.where(y_true[i] == 1)[0].tolist())
          pred_set = set(topk[i].tolist())

          if len(true_set) == 0:
              recalls.append(0)
          else:
              recalls.append(len(true_set & pred_set) / len(true_set))

      return sum(recalls) / len(recalls)

    print("Precision@10:", precision_at_k(y_true, y_pred, 10))
    print("Recall@10:", recall_at_k(y_true, y_pred, 10))

    print("Precision@20:", precision_at_k(y_true, y_pred, 20))
    print("Recall@20:", recall_at_k(y_true, y_pred, 20))

Mounted at /content/drive
No config path provided, using default config


INFO:pyhealth.datasets.mimic3:No config path provided, using default config


Initializing mimic3 dataset from /content/drive/MyDrive/mimic-iii-clinical-database-1.4 (dev mode: False)


INFO:pyhealth.datasets.base_dataset:Initializing mimic3 dataset from /content/drive/MyDrive/mimic-iii-clinical-database-1.4 (dev mode: False)


Using provided cache_dir: /tmp/tmpd3q5poct/c5d69790-6b6e-58a8-abf2-1d953ca43f6a


...

New best pr_auc_samples score (0.4245) at epoch-0, step-353


INFO:pyhealth.trainer:New best pr_auc_samples score (0.4245) at epoch-0, step-353


Loaded best model


INFO:pyhealth.trainer:Loaded best model
Evaluation: 100%|██████████| 45/45 [00:05<00:00,  8.58it/s]


{'jaccard_samples': 0.2168130966969754, 'f1_samples': 0.3523793491037902, 'pr_auc_samples': 0.43343087517038453, 'loss': 1.396124553018146}
Precision@10: 0.6050991501416431
Recall@10: 0.18302369036933058
Precision@20: 0.5057011331444758
Recall@20: 0.3015751825243751
